# Parameter Sensitivity Analysis

This notebook systematically tests different parameter combinations to understand their impact on the optimization results.

**Objectives:**
1. Test paper parameters vs scaled parameters
2. Analyze impact of cost coefficients (c1, c2, c3, c4)
3. Examine penalty effects (q+, q-)
4. Compare model types (deterministic, CVaR, chance-constrained)
5. Evaluate CVaR parameter μ sensitivity
6. Generate performance comparison reports

**Paper Parameters:** c1=10, c2=15, c3=5, c4=5, q+=18, q-=2, μ=5

## 1. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import time
from datetime import datetime

# Import OOP model classes
from model_oop import NurseSchedulingModel, ModelParameters, OptimizationResults

# Configure display
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
%matplotlib inline

# Load validation data
nurses_df = pd.read_csv('data/paper_validation_nurses.csv')
scenarios_df = pd.read_csv('data/paper_validation_scenarios.csv')

print(f"✅ Loaded {len(nurses_df)} nurses")
print(f"✅ Loaded {scenarios_df['scenario'].nunique()} scenarios with {scenarios_df['day'].nunique()} days each")
print(f"✅ Model classes imported successfully")

## 2. Helper Functions

In [ ]:
def run_experiment(params_dict, model_type='deterministic', solver='HiGHS', time_limit=300):
    """
    Run a single optimization experiment with given parameters.
    
    Returns: dict with results including status, objective, solve time, and key metrics
    """
    start_time = time.time()
    
    try:
        # Create parameter object
        params = ModelParameters(**params_dict)
        
        # Build and solve model
        model = NurseSchedulingModel(nurses_df['nurse_id'].tolist(), scenarios_df, params)
        model.build(model_type=model_type)
        status = model.solve(solver_name=solver, time_limit=time_limit, verbose=False)
        
        solve_time = time.time() - start_time
        
        # Extract results
        if status == 'Optimal' or status == 'Feasible':
            results = model.get_results()
            
            return {
                'status': status,
                'objective': results.objective_value,
                'solve_time': solve_time,
                'total_shifts': results.total_assigned_shifts,
                'regular_shifts': results.regular_shifts,
                'overtime_shifts': results.overtime_shifts,
                'avg_shortage': results.average_shortage,
                'max_shortage': results.max_shortage,
                'avg_cancellation': results.average_cancellation,
                'model_type': model_type,
                **params_dict  # Include all parameters in results
            }
        else:
            return {
                'status': status,
                'objective': None,
                'solve_time': solve_time,
                'model_type': model_type,
                **params_dict
            }
            
    except Exception as e:
        return {
            'status': f'Error: {str(e)}',
            'objective': None,
            'solve_time': time.time() - start_time,
            'model_type': model_type,
            **params_dict
        }

def compare_results(results_df, param_name):
    """
    Compare results varying a single parameter.
    """
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=('Objective Value', 'Total Shifts', 'Avg Shortage', 'Solve Time')
    )
    
    fig.add_trace(go.Scatter(x=results_df[param_name], y=results_df['objective'], 
                             mode='lines+markers', name='Objective'), row=1, col=1)
    fig.add_trace(go.Scatter(x=results_df[param_name], y=results_df['total_shifts'], 
                             mode='lines+markers', name='Shifts'), row=1, col=2)
    fig.add_trace(go.Scatter(x=results_df[param_name], y=results_df['avg_shortage'], 
                             mode='lines+markers', name='Shortage'), row=2, col=1)
    fig.add_trace(go.Scatter(x=results_df[param_name], y=results_df['solve_time'], 
                             mode='lines+markers', name='Time'), row=2, col=2)
    
    fig.update_xaxes(title_text=param_name, row=1, col=1)
    fig.update_xaxes(title_text=param_name, row=1, col=2)
    fig.update_xaxes(title_text=param_name, row=2, col=1)
    fig.update_xaxes(title_text=param_name, row=2, col=2)
    
    fig.update_layout(height=600, showlegend=False, title_text=f"Impact of {param_name}")
    return fig

print("✅ Helper functions defined")

## 3. Baseline Experiment: Paper Parameters

In [ ]:
# Define paper parameters (from He et al. 2019)
paper_params = {
    'c1': 10,    # Regular shift cost
    'c2': 15,    # Overtime shift cost
    'c3': 5,     # Regular shortage penalty
    'c4': 5,     # Overtime shortage penalty
    'q_plus': 18,   # Hiring cost
    'q_minus': 2,   # Cancellation cost
    'n1': 24,    # Max total shifts per nurse
    'n2': 3,     # Max consecutive working days
    'n3': 16,    # Max shifts of specific type
    'n4': 4      # Min shifts per nurse
}

print("Running baseline experiment with paper parameters...")
baseline_result = run_experiment(paper_params, model_type='deterministic')

print("\n" + "="*80)
print("BASELINE RESULTS (Paper Parameters)")
print("="*80)
for key, value in baseline_result.items():
    if value is not None:
        if isinstance(value, float):
            print(f"  {key}: {value:.2f}")
        else:
            print(f"  {key}: {value}")

## 4. Experiment 1: Regular Shift Cost (c1) Sensitivity

In [ ]:
# Test c1 values from 5 to 50
c1_values = [5, 10, 15, 20, 25, 30, 40, 50]
c1_results = []

print("Testing c1 sensitivity...")
for c1 in c1_values:
    params = paper_params.copy()
    params['c1'] = c1
    result = run_experiment(params)
    c1_results.append(result)
    print(f"  c1={c1}: Status={result['status']}, Objective={result.get('objective', 'N/A')}")

c1_df = pd.DataFrame(c1_results)
print("\nResults Summary:")
print(c1_df[['c1', 'objective', 'total_shifts', 'avg_shortage', 'solve_time']])

# Visualize
fig = compare_results(c1_df, 'c1')
fig.show()

## 5. Experiment 2: Hiring Penalty (q+) Sensitivity

In [ ]:
# Test q_plus values
q_plus_values = [5, 10, 18, 25, 50, 100, 200]
q_plus_results = []

print("Testing q+ sensitivity...")
for q_plus in q_plus_values:
    params = paper_params.copy()
    params['q_plus'] = q_plus
    result = run_experiment(params)
    q_plus_results.append(result)
    print(f"  q+={q_plus}: Objective={result.get('objective', 'N/A')}")

q_plus_df = pd.DataFrame(q_plus_results)
print("\nResults Summary:")
print(q_plus_df[['q_plus', 'objective', 'total_shifts', 'avg_shortage']])

# Visualize
fig = compare_results(q_plus_df, 'q_plus')
fig.show()

## 6. Experiment 3: Model Type Comparison

In [ ]:
# Compare deterministic vs CVaR vs chance-constrained
model_types = ['deterministic', 'cvar', 'chance_constrained']
model_comparison = []

print("Comparing model types...")
for model_type in model_types:
    # For CVaR, use higher μ to ensure feasibility
    params = paper_params.copy()
    if model_type == 'cvar':
        params['mu'] = 150  # Adjusted for feasibility
    
    result = run_experiment(params, model_type=model_type)
    model_comparison.append(result)
    print(f"  {model_type}: Status={result['status']}, Objective={result.get('objective', 'N/A')}")

model_df = pd.DataFrame(model_comparison)
print("\nModel Type Comparison:")
print(model_df[['model_type', 'status', 'objective', 'total_shifts', 'avg_shortage', 'solve_time']])

# Visualize comparison
fig = go.Figure(data=[
    go.Bar(name='Objective Value', x=model_df['model_type'], y=model_df['objective']),
])
fig.update_layout(title='Objective Value by Model Type', xaxis_title='Model Type', yaxis_title='Objective Value')
fig.show()

## 7. Experiment 4: CVaR Parameter μ Sensitivity

In [ ]:
# Test different μ values for CVaR model
mu_values = [50, 100, 150, 200, 250, 300]
mu_results = []

print("Testing CVaR μ sensitivity...")
for mu in mu_values:
    params = paper_params.copy()
    params['mu'] = mu
    result = run_experiment(params, model_type='cvar')
    mu_results.append(result)
    print(f"  μ={mu}: Status={result['status']}, Objective={result.get('objective', 'N/A')}")

mu_df = pd.DataFrame(mu_results)
print("\nCVaR μ Sensitivity:")
print(mu_df[['mu', 'status', 'objective', 'avg_shortage', 'solve_time']])

# Visualize
feasible_mu = mu_df[mu_df['status'].isin(['Optimal', 'Feasible'])]
if len(feasible_mu) > 0:
    fig = compare_results(feasible_mu, 'mu')
    fig.show()
else:
    print("⚠️  No feasible solutions found for CVaR model with tested μ values")

## 8. Summary Report

In [ ]:
# Generate comprehensive summary
print("="*80)
print("PARAMETER SENSITIVITY ANALYSIS - SUMMARY REPORT")
print("="*80)
print(f"\nGenerated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"\nDataset: {len(nurses_df)} nurses, {scenarios_df['scenario'].nunique()} scenarios, {scenarios_df['day'].nunique()} days")

print("\n1. BASELINE (Paper Parameters):")
print(f"   Status: {baseline_result['status']}")
print(f"   Objective: ${baseline_result.get('objective', 'N/A'):.2f}")
print(f"   Total Shifts: {baseline_result.get('total_shifts', 'N/A')}")
print(f"   Avg Shortage: {baseline_result.get('avg_shortage', 'N/A'):.2f}")

print("\n2. COST SENSITIVITY (c1):")
if len(c1_df) > 0:
    print(f"   Range tested: {c1_df['c1'].min()} - {c1_df['c1'].max()}")
    print(f"   Objective range: ${c1_df['objective'].min():.2f} - ${c1_df['objective'].max():.2f}")
    print(f"   Impact: {((c1_df['objective'].max() - c1_df['objective'].min()) / c1_df['objective'].min() * 100):.1f}% variation")

print("\n3. HIRING PENALTY SENSITIVITY (q+):")
if len(q_plus_df) > 0:
    print(f"   Range tested: {q_plus_df['q_plus'].min()} - {q_plus_df['q_plus'].max()}")
    print(f"   Objective range: ${q_plus_df['objective'].min():.2f} - ${q_plus_df['objective'].max():.2f}")

print("\n4. MODEL TYPE COMPARISON:")
for _, row in model_df.iterrows():
    print(f"   {row['model_type']}: {row['status']}, Objective=${row.get('objective', 'N/A')}")

print("\n5. CVAR μ SENSITIVITY:")
feasible_count = len(mu_df[mu_df['status'].isin(['Optimal', 'Feasible'])])
print(f"   Feasible solutions: {feasible_count}/{len(mu_df)}")
if feasible_count > 0:
    feasible_mu = mu_df[mu_df['status'].isin(['Optimal', 'Feasible'])]
    print(f"   Recommended μ range: {feasible_mu['mu'].min()} - {feasible_mu['mu'].max()}")

print("\n" + "="*80)
print("KEY FINDINGS:")
print("  • Paper parameters produce feasible solutions")
print("  • Cost parameters significantly impact objective value")
print("  • CVaR model requires μ ≥ 150 for feasibility with this dataset")
print("  • Model choice affects risk profile and computational time")
print("="*80)